# FolkloreRAG: LangGraph + Qdrant RAG + LLM-as-judge evals (Colab free tier)
**Before running:** get a free key at https://console.groq.com/keys, then in Colab click the 🔑 icon (left sidebar),
add a secret named `GROQ_API_KEY`, and toggle "Notebook access" on.
Run cells top to bottom. **This notebook never mounts Google Drive.** Everything is stored in Colab's temporary local disk.
Every LLM call is cached locally; to survive a session reset or a daily rate limit, run the last cell to download ONE backup zip,
and upload that zip next time (set RESTORE = True in cell 2).

In [1]:
!pip -q install langgraph qdrant-client sentence-transformers groq tenacity scipy pandas

In [2]:
# ---------- Setup, config, cached LLM helper ----------
import os, re, json, time, hashlib, getpass
import pandas as pd
from google.colab import userdata, files
import io, zipfile, glob, shutil
from groq import Groq, RateLimitError, APIConnectionError, InternalServerError, APITimeoutError
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

BASE = "/content/folklore_rag"          # local, temporary Colab disk (no Google Drive access needed)
CACHE_DIR = f"{BASE}/llm_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

RESTORE = False   # set True to upload a folklore_backup.zip from a previous session (you pick that single file)
if RESTORE:
    up = files.upload()
    for name, data in up.items():
        zipfile.ZipFile(io.BytesIO(data)).extractall(BASE)
    print("Restored", len(os.listdir(CACHE_DIR)), "cached LLM calls")

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = getpass.getpass("Paste your Groq API key: ")

RATE_SLEEP = 3.0   # seconds between uncached calls (free tier has requests-per-minute limits)

llm_client = Groq(api_key=GROQ_API_KEY)

# ---- Auto-discover which models YOUR key can use (Groq renames/retires models often) ----
available = sorted(m.id for m in llm_client.models.list().data)
print("Models your key can use:", available)
SKIP = ("whisper", "guard", "tts", "orpheus", "compound", "safeguard", "embed", "playai")
chat_models = [m for m in available if not any(x in m for x in SKIP)]

def pick(*prefs, exclude=()):
    for p in prefs:
        for m in chat_models:
            if p in m and m not in exclude:
                return m
    return None

WRITER_MODEL = pick("gpt-oss-120b") or chat_models[0]                       # writes the story
CRITIC_MODEL = pick("gpt-oss-20b") or WRITER_MODEL                          # in-loop critic
JUDGE_MODEL  = (pick("qwen", "llama", "gemma", "kimi", "mistral", exclude=(WRITER_MODEL, CRITIC_MODEL))
                or WRITER_MODEL)                                            # final judge: prefer a DIFFERENT model family
# To override manually, uncomment and edit:
# WRITER_MODEL, CRITIC_MODEL, JUDGE_MODEL = "openai/gpt-oss-120b", "openai/gpt-oss-20b", "<another-model-id>"
print("WRITER:", WRITER_MODEL, "| CRITIC:", CRITIC_MODEL, "| JUDGE:", JUDGE_MODEL)
if "gpt-oss" in JUDGE_MODEL:
    print("WARNING: no non-GPT-OSS model is available, so the judge is the same family as the writer. "
          "Mention this limitation (self-preference bias) in your write-up.")

REASONING = ("gpt-oss", "qwen")   # reasoning models spend part of max_tokens on hidden thinking

# Retry only on temporary errors (rate limits, network, 5xx). A wrong model name fails immediately.
@retry(retry=retry_if_exception_type((RateLimitError, APIConnectionError, InternalServerError, APITimeoutError)),
       wait=wait_exponential(multiplier=2, min=5, max=90), stop=stop_after_attempt(8), reraise=True)
def _call(model, system, user, temperature, max_tokens):
    kwargs = {}
    if "gpt-oss" in model:
        kwargs["reasoning_effort"] = "low"
    if any(x in model for x in REASONING):
        max_tokens = max(max_tokens, 2000)
    r = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=temperature,
        max_tokens=max_tokens,
        **kwargs,
    )
    return r.choices[0].message.content or ""

def llm(model, system, user, temperature=0.7, max_tokens=700):
    key = hashlib.sha256(json.dumps([model, system, user, temperature, max_tokens]).encode()).hexdigest()
    path = f"{CACHE_DIR}/{key}.json"
    if os.path.exists(path):
        return json.load(open(path))["out"]
    out = ""
    for mult in (1, 2):   # if a reasoning model ran out of tokens and returned nothing, retry with more room
        out = _call(model, system, user, temperature, max_tokens * mult)
        out = re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip()
        time.sleep(RATE_SLEEP)
        if out:
            break
    if not out:
        raise RuntimeError(f"Model {model} returned empty output twice")
    json.dump({"out": out}, open(path, "w"))
    return out

def parse_json(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

print(llm(WRITER_MODEL, "You are concise.", "Say hello in Hindi in one line.", 0.2, 200))

Models your key can use: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']
WRITER: openai/gpt-oss-120b | CRITIC: openai/gpt-oss-20b | JUDGE: qwen/qwen3.8-27b
नमस्ते!


In [3]:
# ---------- Corpus (short summaries of public-domain Indian tales) ----------
# Tip: add more stories here (or load .txt files you upload with files.upload()) to make retrieval more interesting. 50+ is ideal.
CORPUS = [
 {"title": "The Monkey and the Crocodile", "source": "Panchatantra",
  "text": "A monkey on a jamun tree befriends a crocodile who eats his fruit. The crocodile's wife craves the monkey's heart, so he lures the monkey onto his back. Midstream he confesses. The monkey says he left his heart on the tree and must fetch it, escapes, and never trusts the crocodile again."},
 {"title": "The Lion and the Hare", "source": "Panchatantra",
  "text": "A tyrant lion kills animals every day, so the animals agree to send one voluntarily each day. When it is the clever hare's turn, she arrives late and claims another lion stole her. She leads the enraged lion to a deep well where he sees his own reflection, leaps in to attack it, and drowns."},
 {"title": "The Blue Jackal", "source": "Panchatantra",
  "text": "A jackal falls into an indigo vat and emerges blue. The forest animals take him for a divine ruler and he starts to despise other jackals. One night he hears jackals howl and cannot resist howling along. His disguise is exposed and the animals chase him out."},
 {"title": "The Brahmin and the Mongoose", "source": "Panchatantra",
  "text": "A Brahmin's wife leaves her baby with their pet mongoose and goes out. The mongoose kills a snake that approaches the baby. The Brahmin returns, sees blood on the mongoose's mouth, assumes the worst and kills it. Then he finds the dead snake and the safe child, and is crushed with regret."},
 {"title": "The Crows and the Snake", "source": "Panchatantra",
  "text": "A snake keeps eating the eggs of a crow couple living in a banyan tree. The crows steal a queen's gold necklace and drop it near the snake's hole. The palace guards chase the necklace, find the snake and kill it, ending the threat through cleverness rather than force."},
 {"title": "The Four Friends", "source": "Panchatantra",
  "text": "A deer, a crow, a tortoise and a mouse become close friends. When the deer is caught in a hunter's net, the mouse gnaws the ropes. The slow tortoise is caught by the hunter, so the deer plays dead and the crow pecks at him until the hunter drops the tortoise, who is then freed. Friendship across differences saves them all."},
 {"title": "Tenali Rama and the Cats", "source": "Tenali Rama folk tales",
  "text": "The king gifts cows to his courtiers to keep cats and ensure milk for them. Tenali Rama, who cannot afford to feed a cat on milk, gives his cat scalding milk so it learns to fear milk. At inspection every other cat is plump while Rama's cat refuses milk, revealing how foolish the scheme was."},
 {"title": "Birbal's Khichdi", "source": "Akbar-Birbal tales",
  "text": "Emperor Akbar offers a reward to anyone who can stand in a freezing lake all night. A poor man does it by watching a distant palace lamp. Courtiers say the lamp's warmth helped him and deny the reward. Birbal hangs a pot of khichdi high above a small flame, and when it stays raw, Akbar sees the absurdity and rewards the poor man."},
 {"title": "Idgah", "source": "Munshi Premchand",
  "text": "Hamid, a poor orphan raised by his grandmother Amina, goes to the Eid fair with three paise. His friends buy sweets and toys. Hamid buys iron tongs because his grandmother burns her fingers on the griddle. His friends mock him, but he wins them over with sharp arguments, and Amina weeps with love when she sees the gift."},
 {"title": "The Golden Goose", "source": "Jataka tales",
  "text": "A poor man's goose lays one golden feather every day, lifting his family from poverty. His greedy wife grows impatient and plucks out all the feathers at once hoping for a fortune. The feathers turn ordinary and the golden source is lost forever."},
 {"title": "The Monkey King", "source": "Jataka tales",
  "text": "A monkey king makes a bridge of his own body over a river so his troop can escape a human king's archers. A jealous cousin jumps on his back, breaking it. The human king, moved by the monkey king's selflessness, orders him cared for and gives up the hunt."},
 {"title": "The Talkative Tortoise", "source": "Panchatantra",
  "text": "A tortoise is carried through the air by two geese holding a stick in their beaks, on the condition that he stays silent. Passing a village, he hears the crowd mock the strange sight and opens his mouth to reply. He falls to the ground and dies, a victim of his own ego."},
]

In [4]:
# ---------- Embeddings + Qdrant (in-memory local mode) ----------
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

EMB_MODELS = {
    "e5": "intfloat/multilingual-e5-small",
    "minilm": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
}
ACTIVE = "e5"   # embedding model used by the pipeline

if "qclient" not in globals():
    qclient = QdrantClient(":memory:")   # 12-100 stories re-embed in seconds, so no persistence needed

_st = {}
def embed(key, texts, kind="passage"):
    if key not in _st:
        _st[key] = SentenceTransformer(EMB_MODELS[key])
    if key == "e5":
        texts = [f"{kind}: {t}" for t in texts]
    return _st[key].encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist()

def index_corpus(key):
    name = f"stories_{key}"
    if qclient.collection_exists(name):
        qclient.delete_collection(name)
    vecs = embed(key, [f"{s['title']}. {s['text']}" for s in CORPUS])
    qclient.create_collection(collection_name=name,
                              vectors_config=VectorParams(size=len(vecs[0]), distance=Distance.COSINE))
    qclient.upsert(collection_name=name,
                   points=[PointStruct(id=i, vector=v, payload=CORPUS[i]) for i, v in enumerate(vecs)])

def search(key, query, k=3):
    v = embed(key, [query], "query")[0]
    return qclient.query_points(collection_name=f"stories_{key}", query=v, limit=k).points

for k in EMB_MODELS:
    index_corpus(k)
print("Indexed", len(CORPUS), "stories with", list(EMB_MODELS))

Indexed 12 stories with ['e5', 'minilm']


In [5]:
# ---------- Retrieval eval: recall@3 for two embedding models ----------
RETRIEVAL_EVAL = [
 ("a clever rabbit tricks a tyrant lion into jumping into a well", "The Lion and the Hare"),
 ("एक चालाक खरगोश ने शेर को कुएँ में कूदने पर मजबूर किया", "The Lion and the Hare"),
 ("a monkey escapes a crocodile by saying his heart is left on the tree", "The Monkey and the Crocodile"),
 ("a jackal dyed blue is crowned king but gives himself away by howling", "The Blue Jackal"),
 ("a man kills his loyal pet in haste and later learns the truth", "The Brahmin and the Mongoose"),
 ("a poor boy at a fair buys tongs for his grandmother instead of toys", "Idgah"),
 ("emperor refuses reward to a man who stood in cold water, and a witty minister proves a point with cooking", "Birbal's Khichdi"),
 ("a greedy wife plucks every feather of a magical bird", "The Golden Goose"),
 ("a bird talks and falls from the sky while being carried by two friends", "The Talkative Tortoise"),
 ("a king sacrifices his body to make a bridge for his troop", "The Monkey King"),
]
rows = []
for key in EMB_MODELS:
    hits = sum(any(h.payload["title"] == exp for h in search(key, q, 3)) for q, exp in RETRIEVAL_EVAL)
    rows.append({"embedding_model": EMB_MODELS[key], "recall@3": hits / len(RETRIEVAL_EVAL)})
pd.DataFrame(rows)

,embedding_model,recall@3
0,intfloat/multilingual-e5-small,1.0
1,sentence-transformers/paraphrase-multilingual-...,1.0


In [6]:
# ---------- LangGraph pipeline ----------
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

N_SCENES = 4
PASS_SCORE = 7.0       # critic threshold
MAX_REWRITES = 2       # max rewrites per scene
RUBRIC = ["hook", "character_consistency", "pacing", "emotional_payoff", "cultural_authenticity"]

SYS_WRITER = ("You are a scriptwriter for Indian audio dramas and short story series. You write vivid, dialogue-driven scenes "
              "with distinct character voices, concrete sensory detail, emotional stakes and culturally grounded settings. "
              "Avoid cliches and generic morals.")

class State(TypedDict, total=False):
    premise: str
    use_rag: bool
    use_critic: bool
    refs: List[dict]
    outline: List[str]
    idx: int
    current: str
    tries: int
    score: float
    feedback: str
    scenes: List[str]
    scene_scores: List[float]
    story: str
    nearest_title: str
    max_sim: float

def retrieve(s):
    refs = [h.payload for h in search(ACTIVE, s["premise"], 3)] if s["use_rag"] else []
    return {"refs": refs}

def plan(s):
    ref_txt = "\n".join(f"- {r['title']}: {r['text']}" for r in s["refs"]) or "(none)"
    user = (f"Premise: {s['premise']}\n\nReference folktales (use ONLY for structure and tone, do NOT copy their plots):\n{ref_txt}\n\n"
            f"Plan an original story in exactly {N_SCENES} scenes. Each scene is one sentence naming the characters, "
            "the conflict and what changes. The first scene must open with a hook, and the last must land a satisfying twist or payoff.\n"
            'Respond with ONLY JSON: {"scenes": ["...", "..."]}')
    data = parse_json(llm(WRITER_MODEL, SYS_WRITER, user, 0.7, 600)) or {}
    outline = [str(x) for x in data.get("scenes", [])][:N_SCENES]
    while len(outline) < N_SCENES:   # fallback if the model returned too few
        outline.append("Continue the story toward a satisfying ending.")
    return {"outline": outline, "idx": 0, "tries": 0, "scenes": [], "scene_scores": [], "feedback": "", "current": ""}

def write_scene(s):
    i = s["idx"]
    prev = s["scenes"][-1] if s["scenes"] else "(this is the opening scene)"
    revise = ""
    if s["tries"] > 0:
        revise = (f"\n\nA reviewer found this problem with your previous draft, fix it:\n{s['feedback']}\n\n"
                  f"Previous draft:\n{s['current']}")
    user = (f"Premise: {s['premise']}\n\nFull outline:\n" + "\n".join(f"{j+1}. {b}" for j, b in enumerate(s["outline"])) +
            f"\n\nPrevious scene:\n{prev}\n\nWrite scene {i+1} now: {s['outline'][i]}\n"
            "Rules: 120-180 words, mostly dialogue, short narrator lines in [brackets], end on a line that pulls the listener into the next scene. "
            f"Output ONLY the scene text.{revise}")
    txt = llm(WRITER_MODEL, SYS_WRITER, user, 0.8, 500)
    return {"current": txt.strip(), "tries": s["tries"] + 1}

CRITIC_SYS = "You are a strict, experienced story editor for an audio-drama platform. Be critical and specific. Do not inflate scores."
def rubric_prompt(kind, text, extra=""):
    return (f"Rate this {kind} from 1 to 10 on each criterion: hook (does it grab attention), character_consistency (distinct, consistent voices), "
            "pacing, emotional_payoff, cultural_authenticity (feels grounded in Indian life and idiom, not generic).\n"
            f"{extra}\n---\n{text}\n---\n"
            'Respond with ONLY JSON: {"hook": n, "character_consistency": n, "pacing": n, "emotional_payoff": n, '
            '"cultural_authenticity": n, "weakest_issue": "one specific sentence on the biggest problem and how to fix it"}')

def score_from(data):
    vals = [float(data[k]) for k in RUBRIC if data and isinstance(data.get(k), (int, float))]
    return (sum(vals) / len(vals)) if vals else 7.0

def critic(s):
    if not s["use_critic"]:
        return {"score": 10.0, "feedback": ""}
    data = parse_json(llm(CRITIC_MODEL, CRITIC_SYS, rubric_prompt("scene", s["current"]), 0.0, 400))
    return {"score": score_from(data), "feedback": (data or {}).get("weakest_issue", "")}

def route_after_critic(s):
    return "write_scene" if (s["score"] < PASS_SCORE and s["tries"] <= MAX_REWRITES) else "advance"

def advance(s):
    return {"scenes": s["scenes"] + [s["current"]], "scene_scores": s["scene_scores"] + [s["score"]],
            "idx": s["idx"] + 1, "tries": 0, "feedback": ""}

def route_after_advance(s):
    return "write_scene" if s["idx"] < len(s["outline"]) else "originality_check"

def originality_check(s):
    story = "\n\n".join(s["scenes"])
    v = embed(ACTIVE, [story], "passage")[0]
    top = qclient.query_points(collection_name=f"stories_{ACTIVE}", query=v, limit=1).points[0]
    return {"story": story, "nearest_title": top.payload["title"], "max_sim": float(top.score)}

g = StateGraph(State)
for name, fn in [("retrieve", retrieve), ("plan", plan), ("write_scene", write_scene), ("critic", critic),
                 ("advance", advance), ("originality_check", originality_check)]:
    g.add_node(name, fn)
g.set_entry_point("retrieve")
g.add_edge("retrieve", "plan")
g.add_edge("plan", "write_scene")
g.add_edge("write_scene", "critic")
g.add_conditional_edges("critic", route_after_critic, {"write_scene": "write_scene", "advance": "advance"})
g.add_conditional_edges("advance", route_after_advance, {"write_scene": "write_scene", "originality_check": "originality_check"})
g.add_edge("originality_check", END)
app = g.compile()

def run_pipeline(premise, use_rag=True, use_critic=True):
    return app.invoke({"premise": premise, "use_rag": use_rag, "use_critic": use_critic},
                      config={"recursion_limit": 100})

In [7]:
# ---------- Demo run ----------
out = run_pipeline("A greedy merchant is outsmarted by a village girl during a monsoon festival")
print("RETRIEVED:", [r["title"] for r in out["refs"]])
print("OUTLINE:", *out["outline"], sep="\n  - ")
print("\nSCENE SCORES:", [round(x, 1) for x in out["scene_scores"]])
print(f"NEAREST CORPUS STORY: {out['nearest_title']} (cosine {out['max_sim']:.3f})\n")
print(out["story"])

RETRIEVED: ['Idgah', 'The Brahmin and the Mongoose', 'The Monkey and the Crocodile']
OUTLINE:
  - In the monsoon-soaked village of Chandpur, greedy merchant Laxman arrives at the festival with a cart of overpriced silk, promising the townsfolk a miracle cure for rain‑spattered roofs, while the sharp‑tongued girl Meera watches, already scheming to turn his greed against him.
  - Laxman loudly declares that only his 'rain‑blessed' oil can protect homes, and the villagers, desperate for dry shelter, hand over their hard‑earned rupees, but Meera quietly swaps the oil jars with empty ones, planting a note that claims the oil works only when shared.
  - When the storm intensifies, the villagers splash the empty jars on their roofs, laughing at the harmless trick, yet Laxman's reputation crumbles as he is exposed as a charlatan, and the crowd turns its anger toward him, demanding restitution.
  - In the final twist, Meera reveals she kept a single jar of genuine rain‑repellent herb paste, gif

In [8]:
# ---------- LLM-as-judge for final stories ----------
def evaluate_story(story):
    data = parse_json(llm(JUDGE_MODEL, CRITIC_SYS,
                          rubric_prompt("complete short story", story, "Judge the story as a whole."), 0.0, 1500))
    res = {k: (float(data[k]) if data and isinstance(data.get(k), (int, float)) else None) for k in RUBRIC}
    vals = [v for v in res.values() if v is not None]
    res["judge_overall"] = sum(vals) / len(vals) if vals else None
    res["judge_note"] = (data or {}).get("weakest_issue", "")
    return res

print(evaluate_story(out["story"]))

{'hook': 4.0, 'character_consistency': 3.0, 'pacing': 2.0, 'emotional_payoff': 2.0, 'cultural_authenticity': 5.0, 'judge_overall': 3.2, 'judge_note': "The story is a fragmented collection of four disconnected scenes that never resolve the central conflict, leaving the audience with a cliffhanger instead of a narrative arc; fix this by merging the scenes into a single continuous timeline where the villagers' anger leads to a concrete resolution or consequence for Laxman, rather than ending on a vague 'shadowed figure'."}


In [9]:
# ---------- Ablation: RAG x critic ----------
# Start with 5 premises. Free-tier daily token limits are real; because everything is cached, you can
# re-run this cell tomorrow and it continues where it stopped. Add premises to reach 15-20 for the final write-up.
PREMISES = [
    "A greedy merchant is outsmarted by a village girl during a monsoon festival",
    "A boastful king's minister secretly tests the king's pride before the whole court",
    "A poor weaver's son discovers that his loom speaks only the truth",
    "Two rival fishermen are stranded together on a boat during a cyclone",
    "A grandmother's recipe is stolen by a cunning cook in a royal kitchen",
]
VARIANTS = {"no_rag_no_critic": (False, False), "rag_only": (True, False),
            "critic_only": (False, True), "rag_plus_critic": (True, True)}

results = []
for pid, premise in enumerate(PREMISES):
    for vname, (rag, crit) in VARIANTS.items():
        o = run_pipeline(premise, use_rag=rag, use_critic=crit)
        ev = evaluate_story(o["story"])
        results.append({"id": f"{pid}_{vname}", "premise": premise, "variant": vname, "story": o["story"],
                        "max_sim": o["max_sim"], "nearest": o["nearest_title"], **ev})
        print("done", pid, vname, round(ev["judge_overall"] or 0, 2))

df = pd.DataFrame(results)
df.to_csv(f"{BASE}/ablation_results.csv", index=False)
cols = RUBRIC + ["judge_overall", "max_sim"]
df.groupby("variant")[cols].mean().round(2).sort_values("judge_overall", ascending=False)

done 0 no_rag_no_critic 4.4
done 0 rag_only 5.2
done 0 critic_only 4.4
done 0 rag_plus_critic 3.2
done 1 no_rag_no_critic 3.8
done 1 rag_only 4.4
done 1 critic_only 5.6
done 1 rag_plus_critic 4.0
done 2 no_rag_no_critic 3.8
done 2 rag_only 3.2
done 2 critic_only 4.4
done 2 rag_plus_critic 2.8
done 3 no_rag_no_critic 5.2
done 3 rag_only 2.8
done 3 critic_only 5.0
done 3 rag_plus_critic 3.2
done 4 no_rag_no_critic 3.6
done 4 rag_only 4.4
done 4 critic_only 3.8
done 4 rag_plus_critic 4.4


,hook,character_consistency,pacing,emotional_payoff,cultural_authenticity,judge_overall,max_sim
variant,,,,,,,
critic_only,4.8,3.8,5.2,3.6,5.8,4.64,0.82
no_rag_no_critic,4.4,3.4,4.8,3.8,4.4,4.16,0.83
rag_only,4.4,3.8,3.2,3.6,5.0,4.00,0.83
rag_plus_critic,4.0,3.4,2.8,2.6,4.8,3.52,0.83


In [10]:
# ---------- Human labels: do you agree with the judge? ----------
# 1) Export a BLIND, shuffled sheet (no variant names, no judge scores) so your ratings aren't biased.
label_df = df[["id", "premise", "story"]].sample(frac=1, random_state=0).assign(human_score="")
label_df.to_csv(f"{BASE}/to_label.csv", index=False)
files.download(f"{BASE}/to_label.csv")   # downloads just this one file to your computer
print("Rate each story 1-10 in the human_score column (Excel/Sheets), save as labeled.csv, then run the next cell and upload it.")

Rate each story 1-10 in the human_score column (Excel/Sheets), save as labeled.csv, then run the next cell and upload it.


In [11]:
# 2) After labeling: Spearman correlation between you and the judge
from scipy.stats import spearmanr
up = files.upload()   # pick your labeled.csv
lab = pd.read_csv(io.BytesIO(next(iter(up.values())))).dropna(subset=["human_score"])
merged = lab.merge(df[["id", "variant", "judge_overall"]], on="id").dropna(subset=["judge_overall"])
rho, p = spearmanr(merged["human_score"], merged["judge_overall"])
print(f"Spearman rho = {rho:.2f} (p = {p:.3f}, n = {len(merged)})")
print("Your mean score per variant:")
print(merged.groupby("variant")["human_score"].mean().round(2))

Saving to_label.csv to to_label.csv
Spearman rho = nan (p = nan, n = 0)
Your mean score per variant:
Series([], Name: human_score, dtype: float64)


In [12]:
# ---------- Failure analysis: find where the judge and you disagree, or where output is valid but bland ----------
merged["gap"] = merged["judge_overall"] - merged["human_score"]
print("Judge liked it, you didn't:")
display(merged.sort_values("gap", ascending=False).head(3)[["id", "judge_overall", "human_score"]])
print("Highest-similarity stories (possible copying of corpus):")
display(df.sort_values("max_sim", ascending=False).head(3)[["id", "max_sim", "nearest"]])
# Read these stories and write 3-5 short notes: what was wrong, and what you changed (prompt, rubric, threshold). That's your README gold.

Judge liked it, you didn't:


,id,judge_overall,human_score


Highest-similarity stories (possible copying of corpus):


,id,max_sim,nearest
1,0_rag_only,0.837262,Birbal's Khichdi
8,2_no_rag_no_critic,0.836954,Idgah
11,2_rag_plus_critic,0.833999,Idgah


In [13]:
# ---------- Backup: download ONE zip (LLM cache + results) so you can resume later ----------
zip_path = "/content/folklore_backup"
shutil.make_archive(zip_path, "zip", BASE)
files.download(zip_path + ".zip")